# LeetCode 4052 — Cyclically Shift Rows and Columns bằng Python

Link đề: https://leetcode.com/problems/cyclically-shift-rows-and-columns/ · Độ khó: **Easy** · Chủ đề: Array, Matrix, Simulation

Skeleton LeetCode:

```python
class Solution:
    def cyclicShift(self, n: int, grid: List[List[int]], rowShift: List[int], colShift: List[int]) -> List[List[int]]:
        ...
```

Ghi chú: đề và các ví dụ lấy từ đề gốc (Weekly Contest 519, câu 1); mọi đáp án trong notebook đều được kiểm chứng bằng code, kể cả bằng một lời giải độc lập dịch chuyển từng bước một.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: ma trận vuông `n × n`; trước hết **dịch vòng trái** từng hàng `i` đi `rowShift[i]` ô, rồi (trên ma trận kết quả) **dịch vòng lên** từng cột `j` đi `colShift[j]` ô; trả về ma trận cuối.
2. Biết **dịch vòng một dãy** bằng cắt lát (`a[k:] + a[:k]`) và bằng công thức chỉ số `(j - k) % n`; hiểu `%` của Python xử lý số âm đúng.
3. Biết **làm việc với ma trận**: tạo ma trận mới đúng cách (`[[0] * n for _ in range(n)]`), lấy cột bằng `zip(*grid)` và đặt lại.
4. Đi từ **mô phỏng bằng cắt lát hàng + chuyển vị cột** → **công thức chỉ số, hai ma trận (tạm và kết quả)** → **gộp hai bước thành một lượt** → **công thức "kéo" ngược** → **`deque.rotate`**.
5. Tránh các lỗi kinh điển: dùng `[[0] * n] * n` (các hàng là cùng một đối tượng), sai hướng dịch, **dùng chỉ số cột cũ** thay vì cột mới khi áp `colShift`, ghi đè khi đọc, và đảo thứ tự (cột trước hàng sau).

## 2. Đề bài & ví dụ

Cho số nguyên `n`, ma trận `grid` kích thước `n × n`, và hai mảng `rowShift`, `colShift` đều có độ dài `n`:

- `rowShift[i]` là số vị trí **dịch vòng sang trái** của hàng thứ `i`;
- `colShift[j]` là số vị trí **dịch vòng lên trên** của cột thứ `j`.

Trước tiên dịch vòng **từng hàng** theo `rowShift`; sau đó dịch vòng **từng cột** của ma trận vừa thu được theo `colShift`. Trả về ma trận kết quả.

Dịch vòng trái hàng đi `k` vị trí đưa phần tử ở cột `j` tới cột `(j − k + n) % n`. Dịch vòng lên cột đi `k` vị trí đưa phần tử ở hàng `i` tới hàng `(i − k + n) % n`. Các hàng/cột khác giữ nguyên.

**Ví dụ 1**

- Input: `n = 2`, `grid = [[1,2],[3,4]]`, `rowShift = [1,0]`, `colShift = [0,1]`
- Output: `[[2,4],[3,1]]`
- Giải thích: dịch hàng 0 trái 1: `[1,2] -> [2,1]`, ta có `[[2,1],[3,4]]`; dịch cột 1 lên 1: cột `[1,4] -> [4,1]`, ta có `[[2,4],[3,1]]`.

**Ví dụ 2**

- Input: `n = 3`, `grid = [[1,2,3],[4,5,6],[7,8,9]]`, `rowShift = [1,2,0]`, `colShift = [2,2,1]`
- Output: `[[7,8,5],[2,3,9],[6,4,1]]`

**Ràng buộc:** `1 <= n == grid.length == grid[i].length <= 10`; `1 <= grid[i][j] <= 100`; `rowShift.length == colShift.length == n`; `0 <= rowShift[i], colShift[i] < n`.

In [1]:
from typing import List
import random
from collections import deque

examples = [
    (2, [[1, 2], [3, 4]], [1, 0], [0, 1], [[2, 4], [3, 1]]),
    (3, [[1, 2, 3], [4, 5, 6], [7, 8, 9]], [1, 2, 0], [2, 2, 1], [[7, 8, 5], [2, 3, 9], [6, 4, 1]]),
]


def show(grid, title=""):
    if title:
        print(title)
    for row in grid:
        print("   ", row)


for n, grid, rs, cs, expected in examples:
    print(f"n={n}, rowShift={rs}, colShift={cs}, kỳ vọng {expected}")
    show(grid, "  grid ban đầu:")

n=2, rowShift=[1, 0], colShift=[0, 1], kỳ vọng [[2, 4], [3, 1]]
  grid ban đầu:
    [1, 2]
    [3, 4]
n=3, rowShift=[1, 2, 0], colShift=[2, 2, 1], kỳ vọng [[7, 8, 5], [2, 3, 9], [6, 4, 1]]
  grid ban đầu:
    [1, 2, 3]
    [4, 5, 6]
    [7, 8, 9]


## 3. Kiến thức nền

### 3.1. Ma trận là list của list

Ma trận `grid` là một list; mỗi phần tử `grid[i]` là một list (một **hàng**). Phần tử ở hàng `i`, cột `j` là `grid[i][j]`. Ma trận `n × n` có `n` hàng và `n` cột.

In [2]:
grid = [[1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]]
print(grid[0])          # [1, 2, 3]  hàng 0
print(grid[1][2])       # 6          hàng 1, cột 2
print([row[0] for row in grid])   # [1, 4, 7]  cột 0

[1, 2, 3]
6
[1, 4, 7]


### 3.2. Tạo ma trận mới: `[[0] * n for _ in range(n)]`

Để tạo ma trận kết quả `n × n` toàn số 0, ta viết `[[0] * n for _ in range(n)]`. **Đừng** viết `[[0] * n] * n`: lệnh này nhân bản **cùng một list hàng** `n` lần (các hàng là một đối tượng), nên sửa một hàng sẽ sửa tất cả.

In [3]:
n = 3
good = [[0] * n for _ in range(n)]
bad = [[0] * n] * n
good[0][0] = 1
bad[0][0] = 1
print("good:", good)      # [[1, 0, 0], [0, 0, 0], [0, 0, 0]]
print("bad: ", bad)       # [[1, 0, 0], [1, 0, 0], [1, 0, 0]]  <- cả ba hàng cùng bị đổi
print("bad[0] is bad[1]:", bad[0] is bad[1])

good: [[1, 0, 0], [0, 0, 0], [0, 0, 0]]
bad:  [[1, 0, 0], [1, 0, 0], [1, 0, 0]]
bad[0] is bad[1]: True


### 3.3. Dịch vòng một dãy bằng cắt lát

Với list `a` và `k` trong `0..len(a)-1`, `a[k:] + a[:k]` là dãy **dịch vòng trái `k` vị trí**: `k` phần tử đầu chuyển xuống cuối. Ví dụ `[1,2,3,4]` dịch trái 1 được `[2,3,4,1]`, dịch trái 3 được `[4,1,2,3]`.

In [4]:
a = [1, 2, 3, 4]
for k in range(len(a)):
    print(f"dịch trái {k}: {a[k:] + a[:k]}")

dịch trái 0: [1, 2, 3, 4]
dịch trái 1: [2, 3, 4, 1]
dịch trái 2: [3, 4, 1, 2]
dịch trái 3: [4, 1, 2, 3]


### 3.4. Công thức chỉ số với `%`: phần tử `j` đi đâu?

Khi dịch vòng trái `k` vị trí, phần tử ở chỉ số `j` đến chỉ số mới `(j − k) % n`. Với `j = 0, k = 1, n = 4`: `(0 − 1) % 4 = 3` — đầu dãy nhảy xuống cuối. Python cho `%` luôn không âm với số chia dương, nên không cần cộng thêm `n` như đề (`(j − k + n) % n`) — hai cách cho cùng kết quả khi `0 ≤ k < n`.

In [5]:
n, k = 4, 1
a = [1, 2, 3, 4]
b = [0] * n
for j in range(n):
    b[(j - k) % n] = a[j]               # phần tử j chuyển tới (j - k) % n
print(b)                                  # [2, 3, 4, 1]
assert b == a[k:] + a[:k]
print([(j - k + n) % n for j in range(n)], [(j - k) % n for j in range(n)])   # cùng kết quả

[2, 3, 4, 1]
[3, 0, 1, 2] [3, 0, 1, 2]


### 3.5. Lấy cột: `zip(*grid)` (chuyển vị)

`zip(*grid)` "bung" các hàng làm đối số của `zip`, nên ghép phần tử cùng vị trí của mọi hàng: ta thu được **các cột** (dưới dạng tuple). `list(map(list, zip(*grid)))` biến chúng thành ma trận **chuyển vị** (hàng ↔ cột).

In [6]:
grid = [[1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]]
cols = list(map(list, zip(*grid)))
print(cols)                                   # [[1, 4, 7], [2, 5, 8], [3, 6, 9]]: cột 0, cột 1, cột 2
print(list(map(list, zip(*cols))) == grid)    # True: chuyển vị hai lần trở về ban đầu

[[1, 4, 7], [2, 5, 8], [3, 6, 9]]
True


### 3.6. `collections.deque.rotate`

`deque` là hàng đợi hai đầu; `d.rotate(r)` dịch vòng **sang phải** `r` vị trí (số âm: sang trái), thay đổi tại chỗ. Để dịch **trái** `k` vị trí dùng `d.rotate(-k)`.

In [7]:
d = deque([1, 2, 3, 4])
d.rotate(-1)                     # dịch trái 1
print(list(d))                   # [2, 3, 4, 1]
d.rotate(-3)                     # dịch trái 3 nữa -> tổng 4 = quay đủ vòng
print(list(d))                   # [1, 2, 3, 4]

[2, 3, 4, 1]
[1, 2, 3, 4]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Làm tay ví dụ 1: `grid = [[1,2],[3,4]]`, `rowShift = [1,0]`, `colShift = [0,1]`.

**Bước 1: dịch hàng.** Hàng 0 dịch trái 1: `[1,2] → [2,1]`. Hàng 1 dịch 0: giữ `[3,4]`.

```
[[1, 2],      [[2, 1],
 [3, 4]]  →    [3, 4]]
```

**Bước 2: dịch cột (trên ma trận mới).** Cột 0 dịch 0: giữ `[2,3]`. Cột 1 là `[1,4]`, dịch lên 1: `[4,1]`.

```
[[2, 1],      [[2, 4],
 [3, 4]]  →    [3, 1]]
```

Kết quả `[[2,4],[3,1]]` đúng với đề. Chú ý: ở bước 2 ta dùng **ma trận đã dịch hàng**, không phải ma trận ban đầu; thứ tự **hàng trước, cột sau** rất quan trọng.

In [8]:
grid = [[1, 2], [3, 4]]
rowShift, colShift, n = [1, 0], [0, 1], 2
step1 = [row[k:] + row[:k] for row, k in zip(grid, rowShift)]
show(step1, "sau khi dịch hàng:")
cols = [list(c) for c in zip(*step1)]                          # các cột của ma trận mới
cols = [c[k:] + c[:k] for c, k in zip(cols, colShift)]         # dịch lên = cắt lát giống nhau
step2 = [list(r) for r in zip(*cols)]                           # chuyển vị trở lại
show(step2, "sau khi dịch cột:")

sau khi dịch hàng:
    [2, 1]
    [3, 4]
sau khi dịch cột:
    [2, 4]
    [3, 1]


## 5. Cách 1 — Brute Force / baseline: mô phỏng bằng cắt lát và chuyển vị

**Ý tưởng.** Làm đúng như đề:

1. Với mỗi hàng `i`, thay bằng `row[k:] + row[:k]` với `k = rowShift[i]`.
2. Lấy các cột bằng `zip(*grid)`. Mỗi cột `j` là một list; thay bằng `col[k:] + col[:k]` với `k = colShift[j]` (dịch **lên** cũng là bỏ các phần tử đầu rồi nối xuống cuối, giống dịch trái).
3. Chuyển vị lại để thu được ma trận theo hàng.

Cách này dễ nghĩ vì mỗi dòng code tương ứng một câu trong đề.

In [9]:
def cyclic_shift_slicing(n: int, grid: list[list[int]], rowShift: list[int], colShift: list[int]) -> list[list[int]]:
    # Bước 1: dịch trái từng hàng
    rows = [row[k:] + row[:k] for row, k in zip(grid, rowShift)]
    # Bước 2: dịch lên từng cột (lấy cột bằng zip(*rows))
    cols = [list(col) for col in zip(*rows)]
    cols = [col[k:] + col[:k] for col, k in zip(cols, colShift)]
    # Bước 3: chuyển vị trở lại thành ma trận theo hàng
    return [list(row) for row in zip(*cols)]


for n, grid, rs, cs, expected in examples:
    print(f"n={n} -> {cyclic_shift_slicing(n, grid, rs, cs)} (kỳ vọng {expected})")

n=2 -> [[2, 4], [3, 1]] (kỳ vọng [[2, 4], [3, 1]])
n=3 -> [[7, 8, 5], [2, 3, 9], [6, 4, 1]] (kỳ vọng [[7, 8, 5], [2, 3, 9], [6, 4, 1]])


**Độ phức tạp (Cách 1).** Mỗi hàng và mỗi cột có `n` phần tử, có `n` hàng và `n` cột; mỗi lần cắt lát và nối mất `O(n)`. Tổng **Time `O(n²)`** (số ô của ma trận). **Space `O(n²)`** cho các ma trận trung gian.

**Nút thắt.** Không có nút thắt về thời gian: mọi cách phải chạm vào cả `n²` ô. Hướng cải tiến là **tránh tạo nhiều ma trận trung gian và quan hệ chuyển vị**, bằng cách nói trực tiếp "ô `(i, j)` đi đâu".

## 6. Ý tưởng thuật toán / bước đột phá

**Mỗi phép dịch chỉ là một phép "đổi chỗ" theo công thức chỉ số.** Với hàng `i` dịch trái `rowShift[i]`: phần tử ở `(i, j)` đi tới `(i, (j − rowShift[i]) mod n)`. Với cột `j'` dịch lên `colShift[j']`: phần tử ở `(i, j')` đi tới `((i − colShift[j']) mod n, j')`.

Hai phép dịch **nối tiếp** nên ô `(i, j)` của ma trận gốc đi hai bước:

```
(i, j)  --dịch hàng-->  (i, j')  với j' = (j - rowShift[i]) mod n
        --dịch cột-->   (i', j') với i' = (i - colShift[j']) mod n
```

Chú ý quan trọng: ở bước hai, cột bị dịch là cột **mới** `j'`, nên phải dùng `colShift[j']` (không phải `colShift[j]`). Nhờ vậy ta có thể:

- dùng một ma trận tạm cho phép dịch hàng, rồi một ma trận kết quả cho phép dịch cột (Cách 2);
- hoặc gộp thành **một lượt**, ghi thẳng vào kết quả (Cách 3);
- hoặc đảo ngược: với ô kết quả `(a, b)` tìm xem nó lấy giá trị từ đâu (Cách 4).

In [10]:
n = 3
grid = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
rowShift, colShift = [1, 2, 0], [2, 2, 1]
for i in range(n):
    for j in range(n):
        j2 = (j - rowShift[i]) % n
        i2 = (i - colShift[j2]) % n
        print(f"grid[{i}][{j}]={grid[i][j]}: (i,j)=({i},{j}) -> cột mới j'={j2} -> hàng mới i'={i2}", flush=False)

grid[0][0]=1: (i,j)=(0,0) -> cột mới j'=2 -> hàng mới i'=2
grid[0][1]=2: (i,j)=(0,1) -> cột mới j'=0 -> hàng mới i'=1
grid[0][2]=3: (i,j)=(0,2) -> cột mới j'=1 -> hàng mới i'=1
grid[1][0]=4: (i,j)=(1,0) -> cột mới j'=1 -> hàng mới i'=2
grid[1][1]=5: (i,j)=(1,1) -> cột mới j'=2 -> hàng mới i'=0
grid[1][2]=6: (i,j)=(1,2) -> cột mới j'=0 -> hàng mới i'=2
grid[2][0]=7: (i,j)=(2,0) -> cột mới j'=0 -> hàng mới i'=0
grid[2][1]=8: (i,j)=(2,1) -> cột mới j'=1 -> hàng mới i'=0
grid[2][2]=9: (i,j)=(2,2) -> cột mới j'=2 -> hàng mới i'=1


## 7. Các cách tiếp theo

### Cách 2 — Công thức chỉ số với ma trận tạm (cách của lời giải tham khảo)

**Giải thích.**

1. Tạo ma trận tạm `t` toàn 0. Với mọi `(i, j)`: `t[i][(j - rowShift[i]) % n] = grid[i][j]` (dịch hàng).
2. Tạo ma trận kết quả `ans` toàn 0. Với mọi `(i, j)`: `ans[(i - colShift[j]) % n][j] = t[i][j]` (dịch cột; ở đây `j` đã là cột của `t`, tức cột "mới" của ma trận ban đầu).

In [11]:
def cyclic_shift_two_matrices(n, grid, rowShift, colShift):
    t = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            t[i][(j - rowShift[i]) % n] = grid[i][j]        # dịch hàng: đẩy giá trị sang vị trí mới
    ans = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            ans[(i - colShift[j]) % n][j] = t[i][j]         # dịch cột trên ma trận tạm
    return ans


for n, grid, rs, cs, expected in examples:
    print(f"n={n} -> {cyclic_shift_two_matrices(n, grid, rs, cs)} (kỳ vọng {expected})")

n=2 -> [[2, 4], [3, 1]] (kỳ vọng [[2, 4], [3, 1]])
n=3 -> [[7, 8, 5], [2, 3, 9], [6, 4, 1]] (kỳ vọng [[7, 8, 5], [2, 3, 9], [6, 4, 1]])


In [12]:
# Code nộp LeetCode cho Cách 2
from typing import List


class Solution:
    def cyclicShift(self, n: int, grid: List[List[int]], rowShift: List[int], colShift: List[int]) -> List[List[int]]:
        t = [[0] * n for _ in range(n)]
        for i in range(n):
            for j in range(n):
                t[i][(j - rowShift[i]) % n] = grid[i][j]
        ans = [[0] * n for _ in range(n)]
        for j in range(n):
            for i in range(n):
                ans[(i - colShift[j]) % n][j] = t[i][j]
        return ans

**Độ phức tạp Cách 2:** **Time `O(n²)`** (hai lượt duyệt `n²` ô), **Space `O(n²)`** cho `t` và `ans`.

### Cách 3 — Gộp hai phép dịch thành một lượt

**Giải thích.** Cho mỗi ô `(i, j)` của ma trận gốc, tính cột mới `j2 = (j - rowShift[i]) % n`, rồi hàng mới `i2 = (i - colShift[j2]) % n`, và ghi `ans[i2][j2] = grid[i][j]`. Không cần ma trận tạm `t`. Bắt buộc dùng `colShift[j2]`, không phải `colShift[j]`.

In [13]:
def cyclic_shift_one_pass(n, grid, rowShift, colShift):
    ans = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            j2 = (j - rowShift[i]) % n          # cột sau khi dịch hàng i
            i2 = (i - colShift[j2]) % n         # hàng sau khi dịch cột j2 (cột MỚI)
            ans[i2][j2] = grid[i][j]
    return ans


for n, grid, rs, cs, expected in examples:
    print(f"n={n} -> {cyclic_shift_one_pass(n, grid, rs, cs)} (kỳ vọng {expected})")

n=2 -> [[2, 4], [3, 1]] (kỳ vọng [[2, 4], [3, 1]])
n=3 -> [[7, 8, 5], [2, 3, 9], [6, 4, 1]] (kỳ vọng [[7, 8, 5], [2, 3, 9], [6, 4, 1]])


**Độ phức tạp Cách 3:** **Time `O(n²)`**, **Space `O(n²)`** (chỉ ma trận kết quả; không có ma trận tạm, nhưng đầu ra bắt buộc có `n²` ô).

### Cách 4 — Công thức "kéo" ngược: ô kết quả lấy giá trị từ đâu?

**Giải thích.** Đảo công thức: ô kết quả `(a, b)` nhận giá trị từ ô cột `b` của ma trận tạm, hàng `i = (a + colShift[b]) % n` (vì `a = (i − colShift[b]) mod n`). Ô `(i, b)` của ma trận tạm lấy từ ma trận gốc ở hàng `i`, cột `(b + rowShift[i]) % n`. Ghép lại: `ans[a][b] = grid[i][(b + rowShift[i]) % n]`.

Cách này **mỗi ô kết quả được gán đúng một lần** theo thứ tự `a, b`, rất tiện khi muốn viết bằng list comprehension.

In [14]:
def cyclic_shift_pull(n, grid, rowShift, colShift):
    ans = []
    for a in range(n):
        row = []
        for b in range(n):
            i = (a + colShift[b]) % n                     # hàng nguồn trong ma trận tạm
            row.append(grid[i][(b + rowShift[i]) % n])    # cột nguồn trong ma trận gốc
        ans.append(row)
    return ans


for n, grid, rs, cs, expected in examples:
    print(f"n={n} -> {cyclic_shift_pull(n, grid, rs, cs)} (kỳ vọng {expected})")

n=2 -> [[2, 4], [3, 1]] (kỳ vọng [[2, 4], [3, 1]])
n=3 -> [[7, 8, 5], [2, 3, 9], [6, 4, 1]] (kỳ vọng [[7, 8, 5], [2, 3, 9], [6, 4, 1]])


**Độ phức tạp Cách 4:** **Time `O(n²)`**, **Space `O(n²)`** (chỉ ma trận kết quả).

### Cách 5 — `deque.rotate` cho từng hàng và từng cột

**Giải thích.** Biến mỗi hàng thành `deque`, gọi `rotate(-k)` (dịch trái `k`). Sau đó với mỗi cột, lấy cột bằng list, dựng `deque`, `rotate(-k)` (dịch lên `k` cũng là dịch trái của dãy cột), và ghi trở lại ma trận. Hợp với người quen thao tác `deque`.

In [15]:
def cyclic_shift_deque(n, grid, rowShift, colShift):
    g = [list(row) for row in grid]              # bản sao, không sửa đầu vào
    for i in range(n):
        d = deque(g[i])
        d.rotate(-rowShift[i])                   # dịch trái
        g[i] = list(d)
    for j in range(n):
        d = deque(g[i][j] for i in range(n))     # cột j
        d.rotate(-colShift[j])                   # dịch lên
        for i, v in enumerate(d):
            g[i][j] = v
    return g


for n, grid, rs, cs, expected in examples:
    print(f"n={n} -> {cyclic_shift_deque(n, grid, rs, cs)} (kỳ vọng {expected})")

n=2 -> [[2, 4], [3, 1]] (kỳ vọng [[2, 4], [3, 1]])
n=3 -> [[7, 8, 5], [2, 3, 9], [6, 4, 1]] (kỳ vọng [[7, 8, 5], [2, 3, 9], [6, 4, 1]])


**Độ phức tạp Cách 5:** **Time `O(n²)`**, **Space `O(n²)`** (bản sao ma trận và `deque` tạm `O(n)`).

> Cách 2 trùng với lời giải tham khảo; Cách 1 dễ đọc nhất; Cách 3 và 4 ngắn gọn khi đã quen công thức chỉ số.

## 8. TRACE từng bước

Ta theo dõi Cách 3 (gộp một lượt) trên ví dụ 2: `n = 3`, `rowShift = [1,2,0]`, `colShift = [2,2,1]`. Với mỗi ô gốc `(i, j)`, in cột mới `j'`, hàng mới `i'`, và ma trận kết quả đang được lấp dần.

In [16]:
def trace_one_pass(n, grid, rowShift, colShift):
    ans = [[0] * n for _ in range(n)]
    print(f"n={n}, rowShift={rowShift}, colShift={colShift}")
    for i in range(n):
        for j in range(n):
            j2 = (j - rowShift[i]) % n
            i2 = (i - colShift[j2]) % n
            ans[i2][j2] = grid[i][j]
            print(f"  gốc ({i},{j})={grid[i][j]}: j'=({j}-{rowShift[i]})%{n}={j2}, i'=({i}-{colShift[j2]})%{n}={i2} -> ans[{i2}][{j2}]={grid[i][j]}")
    show(ans, "  kết quả:")
    return ans


n, grid, rs, cs, expected = examples[1]
assert trace_one_pass(n, grid, rs, cs) == expected

n=3, rowShift=[1, 2, 0], colShift=[2, 2, 1]
  gốc (0,0)=1: j'=(0-1)%3=2, i'=(0-1)%3=2 -> ans[2][2]=1
  gốc (0,1)=2: j'=(1-1)%3=0, i'=(0-2)%3=1 -> ans[1][0]=2
  gốc (0,2)=3: j'=(2-1)%3=1, i'=(0-2)%3=1 -> ans[1][1]=3
  gốc (1,0)=4: j'=(0-2)%3=1, i'=(1-2)%3=2 -> ans[2][1]=4
  gốc (1,1)=5: j'=(1-2)%3=2, i'=(1-1)%3=0 -> ans[0][2]=5
  gốc (1,2)=6: j'=(2-2)%3=0, i'=(1-2)%3=2 -> ans[2][0]=6
  gốc (2,0)=7: j'=(0-0)%3=0, i'=(2-2)%3=0 -> ans[0][0]=7
  gốc (2,1)=8: j'=(1-0)%3=1, i'=(2-2)%3=0 -> ans[0][1]=8
  gốc (2,2)=9: j'=(2-0)%3=2, i'=(2-1)%3=1 -> ans[1][2]=9
  kết quả:
    [7, 8, 5]
    [2, 3, 9]
    [6, 4, 1]


In [17]:
def trace_two_steps(n, grid, rowShift, colShift):
    print("Ma trận gốc:")
    show(grid)
    rows = [row[k:] + row[:k] for row, k in zip(grid, rowShift)]
    for i in range(n):
        print(f"  hàng {i} dịch trái {rowShift[i]}: {grid[i]} -> {rows[i]}")
    show(rows, "Sau khi dịch hàng:")
    cols = [list(c) for c in zip(*rows)]
    new_cols = [c[k:] + c[:k] for c, k in zip(cols, colShift)]
    for j in range(n):
        print(f"  cột {j} dịch lên {colShift[j]}: {cols[j]} -> {new_cols[j]}")
    result = [list(r) for r in zip(*new_cols)]
    show(result, "Sau khi dịch cột (kết quả):")
    return result


n, grid, rs, cs, expected = examples[1]
assert trace_two_steps(n, grid, rs, cs) == expected

Ma trận gốc:
    [1, 2, 3]
    [4, 5, 6]
    [7, 8, 9]
  hàng 0 dịch trái 1: [1, 2, 3] -> [2, 3, 1]
  hàng 1 dịch trái 2: [4, 5, 6] -> [6, 4, 5]
  hàng 2 dịch trái 0: [7, 8, 9] -> [7, 8, 9]
Sau khi dịch hàng:
    [2, 3, 1]
    [6, 4, 5]
    [7, 8, 9]
  cột 0 dịch lên 2: [2, 6, 7] -> [7, 2, 6]
  cột 1 dịch lên 2: [3, 4, 8] -> [8, 3, 4]
  cột 2 dịch lên 1: [1, 5, 9] -> [5, 9, 1]
Sau khi dịch cột (kết quả):
    [7, 8, 5]
    [2, 3, 9]
    [6, 4, 1]


Bảng tóm tắt (ví dụ 2):

| bước | ma trận |
|---|---|
| gốc | `[[1,2,3],[4,5,6],[7,8,9]]` |
| dịch hàng (`[1,2,0]`) | `[[2,3,1],[6,4,5],[7,8,9]]` |
| dịch cột (`[2,2,1]`) | `[[7,8,5],[2,3,9],[6,4,1]]` |

Ví dụ ô `(0, 0) = 1`: dịch hàng 0 trái 1 đưa nó tới cột `(0-1) % 3 = 2`; rồi cột 2 dịch lên 1 đưa nó tới hàng `(0-1) % 3 = 2`; vậy `1` kết thúc ở `(2, 2)`, đúng với đáp án (góc dưới phải là `1`).

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (gộp một lượt, không cần ma trận tạm). Đúng class `Solution`, đúng method `cyclicShift(self, n: int, grid: List[List[int]], rowShift: List[int], colShift: List[int]) -> List[List[int]]`.

In [18]:
from typing import List


class Solution:
    def cyclicShift(self, n: int, grid: List[List[int]], rowShift: List[int], colShift: List[int]) -> List[List[int]]:
        ans = [[0] * n for _ in range(n)]
        for i in range(n):
            for j in range(n):
                j2 = (j - rowShift[i]) % n          # cột mới sau khi dịch hàng i sang trái
                i2 = (i - colShift[j2]) % n         # hàng mới sau khi dịch cột j2 lên trên
                ans[i2][j2] = grid[i][j]
        return ans


for n, grid, rs, cs, expected in examples:
    assert Solution().cyclicShift(n, grid, rs, cs) == expected
print("PASS ví dụ đề cho class Solution")

PASS ví dụ đề cho class Solution


## 10. Vì sao thuật toán đúng

- **Phép dịch là song ánh (hoán vị) các ô:** dịch vòng không mất, không nhân đôi phần tử; mỗi ô gốc đi tới đúng một ô và mỗi ô đích nhận đúng một phần tử. Nên `ans` không có ô bị bỏ trống (giữ `0`) hay bị ghi đè.
- **Công thức đúng theo đề:** đề cho dịch trái hàng: cột `j → (j − k + n) % n`; khi `0 ≤ k < n` thì `(j − k + n) % n == (j − k) % n` với `%` của Python. Tương tự cho dịch lên.
- **Thứ tự hai bước:** hàng trước, cột sau. Sau khi dịch hàng, phần tử gốc `(i, j)` đang ở cột `j' = (j − rowShift[i]) % n`; phép dịch cột **chỉ phụ thuộc cột hiện tại `j'`**, nên áp dụng `colShift[j']` rồi tính hàng mới `(i − colShift[j']) % n` (hàng trong ma trận tạm vẫn là `i` vì dịch hàng không đổi hàng).
- **Cách 4 đúng vì** là nghịch đảo của hai công thức đẩy: `i' = (i − c) % n ⇔ i = (i' + c) % n` và `j' = (j − r) % n ⇔ j = (j' + r) % n`.
- **Kiểm chứng:** Cách 1 (cắt lát) đúng nghĩa đen theo đề, và mọi cách khác được đối chiếu với nó trong phần test.

## 11. Phân tích Time / Space Complexity

Gọi `n` là cạnh ma trận (đề cho `n ≤ 10`); có `n²` ô.

- **Cách 1 (cắt lát + chuyển vị):** Time `O(n²)`, Space `O(n²)`.
- **Cách 2 (ma trận tạm + ma trận kết quả):** Time `O(n²)`, Space `O(n²)`.
- **Cách 3 (một lượt):** Time `O(n²)`, Space `O(n²)` (chỉ đầu ra).
- **Cách 4 (kéo ngược):** Time `O(n²)`, Space `O(n²)` (chỉ đầu ra).
- **Cách 5 (`deque.rotate`):** Time `O(n²)`, Space `O(n²)`.
- Mọi cách phải ghi `n²` ô nên không thể tốt hơn `O(n²)`; trung bình và xấu nhất như nhau.

## 12. Bộ test

Ta gom các cách vào từ điển `impls`, và viết một **lời giải độc lập** `oracle` dịch **từng bước một**: lặp `k` lần phép dịch 1 đơn vị (đổi chỗ phần tử qua từng ô), hoàn toàn khác công thức `%`/cắt lát.

In [19]:
sol = Solution()
impls = {
    "slicing": cyclic_shift_slicing,
    "two_matrices": cyclic_shift_two_matrices,
    "one_pass": cyclic_shift_one_pass,
    "pull": cyclic_shift_pull,
    "deque": cyclic_shift_deque,
    "submit": sol.cyclicShift,
}


def oracle(n, grid, rowShift, colShift):
    """Mô phỏng từng bước 1 đơn vị: mỗi lần dịch trái 1 ô bằng cách hoán đổi lần lượt cặp kề nhau."""
    g = [list(r) for r in grid]
    for i in range(n):
        for _ in range(rowShift[i]):
            first = g[i][0]
            for j in range(n - 1):
                g[i][j] = g[i][j + 1]
            g[i][n - 1] = first
    for j in range(n):
        for _ in range(colShift[j]):
            first = g[0][j]
            for i in range(n - 1):
                g[i][j] = g[i + 1][j]
            g[n - 1][j] = first
    return g


def run(fn, n, grid, rs, cs):
    # truyền bản sao để phát hiện hàm sửa đầu vào
    return fn(n, [list(r) for r in grid], list(rs), list(cs))


for n, grid, rs, cs, expected in examples:
    assert oracle(n, grid, rs, cs) == expected
    for name, fn in impls.items():
        got = run(fn, n, grid, rs, cs)
        assert got == expected, (name, n, grid, rs, cs, got, expected)
print("PASS ví dụ của đề (mọi cách và oracle độc lập)")

PASS ví dụ của đề (mọi cách và oracle độc lập)


### 12.1. Các trường hợp biên

In [20]:
def check_all(n, grid, rs, cs, expected, label):
    assert oracle(n, grid, rs, cs) == expected, (label, "oracle", oracle(n, grid, rs, cs), expected)
    for name, fn in impls.items():
        got = run(fn, n, grid, rs, cs)
        assert got == expected, (label, name, got, expected)
    print("PASS", label)


check_all(1, [[5]], [0], [0], [[5]], "n = 1")
check_all(2, [[1, 2], [3, 4]], [0, 0], [0, 0], [[1, 2], [3, 4]], "không dịch gì")
check_all(2, [[1, 2], [3, 4]], [1, 1], [0, 0], [[2, 1], [4, 3]], "chỉ dịch hàng, n = 2")
check_all(2, [[1, 2], [3, 4]], [0, 0], [1, 1], [[3, 4], [1, 2]], "chỉ dịch cột, n = 2")
check_all(3, [[1, 2, 3], [4, 5, 6], [7, 8, 9]], [2, 2, 2], [0, 0, 0], [[3, 1, 2], [6, 4, 5], [9, 7, 8]], "dịch trái n - 1 mỗi hàng")
check_all(3, [[1, 2, 3], [4, 5, 6], [7, 8, 9]], [0, 0, 0], [1, 1, 1], [[4, 5, 6], [7, 8, 9], [1, 2, 3]], "dịch cột lên 1 = dịch hàng lên")
check_all(3, [[7, 7, 7]] * 3, [1, 2, 0], [2, 2, 1], [[7, 7, 7]] * 3, "mọi ô bằng nhau")
check_all(2, [[1, 2], [3, 4]], [1, 0], [0, 1], [[2, 4], [3, 1]], "ví dụ 1 (đề)")

PASS n = 1
PASS không dịch gì
PASS chỉ dịch hàng, n = 2
PASS chỉ dịch cột, n = 2
PASS dịch trái n - 1 mỗi hàng
PASS dịch cột lên 1 = dịch hàng lên
PASS mọi ô bằng nhau
PASS ví dụ 1 (đề)


### 12.2. Tính chất và phản ví dụ

In [21]:
random.seed(4052)


def random_case(n):
    grid = [[random.randint(1, 100) for _ in range(n)] for _ in range(n)]
    rs = [random.randrange(n) for _ in range(n)]
    cs = [random.randrange(n) for _ in range(n)]
    return grid, rs, cs


# Tính chất 1: kết quả là hoán vị của các ô (cùng tập giá trị, cùng tổng)
for _ in range(300):
    n = random.randint(1, 10)
    grid, rs, cs = random_case(n)
    out = sol.cyclicShift(n, grid, rs, cs)
    assert sorted(sum(grid, [])) == sorted(sum(out, []))
print("PASS tính chất: kết quả là hoán vị các giá trị gốc")

# Tính chất 2: chỉ dịch hàng -> mỗi hàng là một phép dịch vòng của hàng gốc
for _ in range(200):
    n = random.randint(1, 8)
    grid, rs, _cs = random_case(n)
    out = sol.cyclicShift(n, grid, rs, [0] * n)
    for i in range(n):
        assert out[i] == grid[i][rs[i]:] + grid[i][:rs[i]]
print("PASS tính chất: không dịch cột thì chỉ có hàng bị dịch")

# Tính chất 3: ngược lại, mọi hàng trùng nhau trước thì mọi cột đúng bằng cột đã dịch của cột gốc khi không dịch hàng
for _ in range(200):
    n = random.randint(1, 8)
    grid, _rs, cs = random_case(n)
    out = sol.cyclicShift(n, grid, [0] * n, cs)
    for j in range(n):
        col = [grid[i][j] for i in range(n)]
        assert [out[i][j] for i in range(n)] == col[cs[j]:] + col[:cs[j]]
print("PASS tính chất: không dịch hàng thì chỉ có cột bị dịch")

# Phản ví dụ: đảo thứ tự (cột trước, hàng sau) cho kết quả khác
n, grid, rs, cs, expected = examples[1]
def sai_dao_thu_tu(n, grid, rs, cs):
    g = [list(r) for r in grid]
    cols = [list(c) for c in zip(*g)]
    cols = [c[k:] + c[:k] for c, k in zip(cols, cs)]
    g = [list(r) for r in zip(*cols)]
    return [r[k:] + r[:k] for r, k in zip(g, rs)]

print("cột trước hàng sau:", sai_dao_thu_tu(n, grid, rs, cs), "| đúng:", expected)
assert sai_dao_thu_tu(n, grid, rs, cs) != expected

PASS tính chất: kết quả là hoán vị các giá trị gốc
PASS tính chất: không dịch cột thì chỉ có hàng bị dịch
PASS tính chất: không dịch hàng thì chỉ có cột bị dịch
cột trước hàng sau: [[8, 6, 7], [9, 1, 2], [4, 5, 3]] | đúng: [[7, 8, 5], [2, 3, 9], [6, 4, 1]]


### 12.3. Đối chiếu ngẫu nhiên (có seed) với oracle độc lập

In [22]:
random.seed(2026)
for trial in range(2000):
    n = random.randint(1, 10)
    grid, rs, cs = random_case(n)
    expected = oracle(n, grid, rs, cs)
    for name, fn in impls.items():
        got = run(fn, n, grid, rs, cs)
        assert got == expected, (name, n, grid, rs, cs)
print("PASS 2000 input ngẫu nhiên: mọi cách khớp oracle độc lập (và không sửa đầu vào)")

PASS 2000 input ngẫu nhiên: mọi cách khớp oracle độc lập (và không sửa đầu vào)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Tạo ma trận bằng `[[0] * n] * n`

In [23]:
def sai_alias_hang(n, grid, rowShift, colShift):
    t = [[0] * n] * n                    # n hàng cùng là MỘT list
    for i in range(n):
        for j in range(n):
            t[i][(j - rowShift[i]) % n] = grid[i][j]
    ans = [[0] * n for _ in range(n)]
    for j in range(n):
        for i in range(n):
            ans[(i - colShift[j]) % n][j] = t[i][j]
    return ans


n, grid, rs, cs, expected = examples[0]
print("sai:", sai_alias_hang(n, grid, rs, cs), "| đúng:", expected)

sai: [[3, 4], [3, 4]] | đúng: [[2, 4], [3, 1]]


**Vì sao sai:** `[[0] * n] * n` tạo `n` tham chiếu tới **cùng một** list hàng. Ghi vào `t[0]` cũng là ghi vào `t[1]`, nên các hàng ghi chồng lên nhau và dữ liệu bị mất. Luôn dùng `[[0] * n for _ in range(n)]`.

### Lỗi 2: Dùng chỉ số cột cũ `colShift[j]` thay vì cột mới `colShift[j2]`

In [24]:
def sai_cot_cu(n, grid, rowShift, colShift):
    ans = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            j2 = (j - rowShift[i]) % n
            i2 = (i - colShift[j]) % n          # sai: phải là colShift[j2]
            ans[i2][j2] = grid[i][j]
    return ans


n, grid, rs, cs, expected = examples[1]
print("sai:", sai_cot_cu(n, grid, rs, cs), "| đúng:", expected)

sai: [[7, 8, 0], [2, 0, 9], [0, 4, 5]] | đúng: [[7, 8, 5], [2, 3, 9], [6, 4, 1]]


**Vì sao sai:** sau khi dịch hàng, phần tử gốc `(i, j)` đã nằm ở cột `j2`, và phép dịch cột áp dụng cho **cột hiện tại** `j2`. `colShift[j]` là dịch của cột `j` (một cột khác). Hậu quả: các phần tử chuyển nhầm hàng, thậm chí ghi chồng lên nhau (ô trống `0` xuất hiện).

### Lỗi 3: Đọc và ghi cùng một ma trận (ghi đè khi đang đọc)

In [25]:
def sai_ghi_de(n, grid, rowShift, colShift):
    g = grid                                    # không sao chép: sửa trực tiếp đầu vào
    for i in range(n):
        for j in range(n):
            g[i][(j - rowShift[i]) % n] = g[i][j]       # ghi lên ô sẽ còn phải đọc
    return g


n, grid, rs, cs, expected = examples[0]
print("sai:", sai_ghi_de(n, [list(r) for r in grid], rs, cs), "| dịch hàng đúng:", [r[k:] + r[:k] for r, k in zip(grid, rs)])

sai: [[1, 1], [3, 4]] | dịch hàng đúng: [[2, 1], [3, 4]]


**Vì sao sai:** khi dịch tại chỗ bằng cách gán thẳng, ô `(i, (j − k) % n)` bị ghi đè **trước khi** giá trị cũ của nó được đọc để chuyển đi. Hàng `[1, 2]` dịch trái 1 thành `[1, 1]` (mất `2`). Cần ma trận mới (hoặc ghi qua biến tạm đúng chu trình). Hai ma trận (`t`, `ans`) trong Cách 2 chính vì lý do này.

### Lỗi 4: Dịch ngược hướng (dịch phải/xuống thay vì trái/lên)

In [26]:
def sai_huong(n, grid, rowShift, colShift):
    ans = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            j2 = (j + rowShift[i]) % n          # sai: dịch phải
            i2 = (i + colShift[j2]) % n         # sai: dịch xuống
            ans[i2][j2] = grid[i][j]
    return ans


n, grid, rs, cs, expected = examples[0]
print("sai:", sai_huong(n, grid, rs, cs), "| đúng:", expected)
n, grid, rs, cs, expected = examples[1]
print("sai:", sai_huong(n, grid, rs, cs), "| đúng:", expected)

sai: [[2, 4], [3, 1]] | đúng: [[2, 4], [3, 1]]
sai: [[5, 6, 9], [7, 8, 2], [3, 1, 4]] | đúng: [[7, 8, 5], [2, 3, 9], [6, 4, 1]]


**Vì sao sai:** đề quy định dịch **trái** và **lên**, nên chỉ số mới là `(j − k) % n`. Dấu `+` đưa phần tử đi hướng ngược lại (ở ví dụ 1 hai hướng trùng nhau vì `n = 2`, nhưng ví dụ 2 với `n = 3` thì khác). Luôn thử cả hai ví dụ, nhất là ví dụ `n ≥ 3`.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "dịch vòng / cyclic shift / rotate" một dãy: nghĩ đến **cắt lát `a[k:] + a[:k]`**, **`deque.rotate`** hoặc **công thức `(j − k) % n`**.
- Bài ma trận với `n ≤ 10`: **mô phỏng đúng đề** là hướng an toàn nhất; viết cách cắt lát trước, rồi mới tối ưu bằng công thức.
- Cần làm hai bước nối tiếp lên cùng một ma trận: nghĩ xem phần tử "đang ở đâu" sau bước một rồi mới áp bước hai (dùng chỉ số **mới**).
- Tạo ma trận mới luôn là `[[0] * n for _ in range(n)]`. Muốn sao chép ma trận: `[row[:] for row in grid]` hoặc `[list(r) for r in grid]`.
- `zip(*grid)` là cách ngắn nhất để lấy cột/chuyển vị.
- Luôn kiểm tra **cả hai ví dụ** và một ví dụ `n = 3`: các lỗi hướng và thứ tự thường lộ ra khi `n ≥ 3`.

## 15. Biến thể của bài

1. **Dịch phải và xuống** thay vì trái và lên (đảo dấu công thức).
2. **Cột trước, hàng sau** (đổi thứ tự thực hiện).
3. **Khôi phục** ma trận gốc từ kết quả và hai mảng dịch (làm ngược).
4. **Ma trận chữ nhật** `m × n` (dịch hàng theo `n`, dịch cột theo `m`).
5. **Dịch cả ma trận như một dãy phẳng** (dịch vòng toàn bộ `n²` phần tử) — bài LeetCode "Shift 2D Grid".

In [27]:
def cyclic_shift_cols_first(n, grid, rowShift, colShift):
    """Biến thể 2: dịch cột lên trước, rồi dịch hàng trái."""
    cols = [list(c) for c in zip(*grid)]
    cols = [c[k:] + c[:k] for c, k in zip(cols, colShift)]
    g = [list(r) for r in zip(*cols)]
    return [r[k:] + r[:k] for r, k in zip(g, rowShift)]


def cyclic_unshift(n, result, rowShift, colShift):
    """Biến thể 3: khôi phục ma trận gốc (hoàn tác cột rồi hoàn tác hàng)."""
    cols = [list(c) for c in zip(*result)]
    cols = [c[-k:] + c[:-k] if k else c for c, k in zip(cols, colShift)]    # dịch xuống k = hoàn tác dịch lên k
    g = [list(r) for r in zip(*cols)]
    return [r[-k:] + r[:-k] if k else r for r, k in zip(g, rowShift)]       # dịch phải k = hoàn tác dịch trái k


def cyclic_shift_rect(m, n, grid, rowShift, colShift):
    """Biến thể 4: ma trận m hàng, n cột."""
    t = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            t[i][(j - rowShift[i]) % n] = grid[i][j]
    ans = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            ans[(i - colShift[j]) % m][j] = t[i][j]
    return ans


random.seed(15)
for _ in range(200):
    n = random.randint(1, 8)
    grid, rs, cs = random_case(n)
    out = sol.cyclicShift(n, grid, rs, cs)
    assert cyclic_unshift(n, out, rs, cs) == grid
    assert cyclic_shift_rect(n, n, grid, rs, cs) == out
assert cyclic_shift_cols_first(*examples[1][:4]) != examples[1][4]
print(cyclic_shift_rect(2, 3, [[1, 2, 3], [4, 5, 6]], [1, 0], [1, 0, 1]))     # ma trận 2 x 3
print("PASS biến thể: hoàn tác, ma trận chữ nhật, đổi thứ tự")

[[4, 3, 6], [2, 5, 1]]
PASS biến thể: hoàn tác, ma trận chữ nhật, đổi thứ tự


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Cắt lát + chuyển vị | `row[k:] + row[:k]`, `zip(*grid)` | `O(n²)` | `O(n²)` | đúng nghĩa đen theo đề; đối chiếu |
| 2. Hai ma trận (tạm + kết quả) | công thức chỉ số mỗi bước | `O(n²)` | `O(n²)` | **cách của lời giải tham khảo**, dễ đọc |
| 3. Gộp một lượt | `j2 = (j-r)%n`, `i2 = (i-c[j2])%n` | `O(n²)` | `O(n²)` | **cách nộp bài**, không cần ma trận tạm |
| 4. "Kéo" ngược | `ans[a][b] = grid[i][(b+r[i])%n]`, `i=(a+c[b])%n` | `O(n²)` | `O(n²)` | viết bằng comprehension, mỗi ô ghi một lần |
| 5. `deque.rotate` | `rotate(-k)` từng hàng/cột | `O(n²)` | `O(n²)` | khi quen `deque` |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `[1, 2, 3][1:] + [1, 2, 3][:1]`, `(0 - 2) % 3`, `list(zip(*[[1, 2], [3, 4]]))`, `Solution().cyclicShift(1, [[7]], [0], [0])`, và `Solution().cyclicShift(2, [[1, 2], [3, 4]], [0, 0], [1, 0])`.

**Bài 2 (tính tay).** Với `grid = [[1,2,3],[4,5,6],[7,8,9]]` và `rowShift = [1,0,2]`: tính ma trận sau khi dịch hàng. Sau đó với `colShift = [1,0,0]` tính ma trận cuối.

**Bài 3 (điền TODO).** Hoàn thành `shift_todo(n, grid, rowShift, colShift)` theo Cách 3.

In [28]:
def shift_todo(n, grid, rowShift, colShift):
    ans = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            # TODO 1: j2 = cột mới sau khi dịch hàng i sang trái rowShift[i]
            # TODO 2: i2 = hàng mới sau khi dịch cột j2 lên colShift[j2]
            # TODO 3: ans[i2][j2] = grid[i][j]
            pass
    return ans

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `shift_pull2(n, grid, rowShift, colShift)` theo Cách 4 (công thức "kéo" ngược) mà không nhìn code mẫu. Kiểm tra bằng hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `shift_right_down(n, grid, rowShift, colShift)`: dịch hàng sang **phải** rồi dịch cột **xuống**. (b) Viết `unshift(n, result, rowShift, colShift)` khôi phục ma trận gốc từ kết quả (hoàn tác cột trước, hàng sau).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: cắt lát `[1:]` bỏ phần tử đầu; `%` của Python cho số không âm; `zip(*m)` ghép các cột; một ô thì mọi dịch đều không đổi. Bài 2: dịch trái 1 là chuyển phần tử đầu xuống cuối.
- **Mức 2 (vừa):** Bài 1: `[2, 3, 1]`, `1`, `[(1, 3), (2, 4)]`, `[[7]]`, `[[3, 2], [1, 4]]`. Bài 2: dịch hàng: `[[2,3,1],[4,5,6],[9,7,8]]`; cột 0 là `[2,4,9]` dịch lên 1 thành `[4,9,2]`. Bài 3: ba TODO trong comment.
- **Mức 3 (sâu):** Bài 2 kết quả cuối: `[[4,3,1],[9,5,6],[2,7,8]]`. Bài 4: `i = (a + colShift[b]) % n`, `ans[a][b] = grid[i][(b + rowShift[i]) % n]`. Bài 5(a): đổi `-` thành `+` ở cả hai công thức. Bài 5(b): hoàn tác cột bằng dịch **xuống** `colShift[j]` rồi hoàn tác hàng bằng dịch **phải** `rowShift[i]`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [29]:
# Bài 1
assert [1, 2, 3][1:] + [1, 2, 3][:1] == [2, 3, 1]
assert (0 - 2) % 3 == 1
assert list(zip(*[[1, 2], [3, 4]])) == [(1, 3), (2, 4)]
assert Solution().cyclicShift(1, [[7]], [0], [0]) == [[7]]
assert Solution().cyclicShift(2, [[1, 2], [3, 4]], [0, 0], [1, 0]) == [[3, 2], [1, 4]]

# Bài 2
g2 = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
assert Solution().cyclicShift(3, g2, [1, 0, 2], [0, 0, 0]) == [[2, 3, 1], [4, 5, 6], [9, 7, 8]]
assert Solution().cyclicShift(3, g2, [1, 0, 2], [1, 0, 0]) == [[4, 3, 1], [9, 5, 6], [2, 7, 8]]

# Bài 3
def shift_todo_ans(n, grid, rowShift, colShift):
    ans = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            j2 = (j - rowShift[i]) % n
            i2 = (i - colShift[j2]) % n
            ans[i2][j2] = grid[i][j]
    return ans

for n, grid, rs, cs, expected in examples:
    assert shift_todo_ans(n, grid, rs, cs) == expected

# Bài 4
def shift_pull2(n, grid, rowShift, colShift):
    ans = [[0] * n for _ in range(n)]
    for a in range(n):
        for b in range(n):
            i = (a + colShift[b]) % n
            ans[a][b] = grid[i][(b + rowShift[i]) % n]
    return ans

for n, grid, rs, cs, expected in examples:
    assert shift_pull2(n, grid, rs, cs) == expected

# Bài 5(a)
def shift_right_down(n, grid, rowShift, colShift):
    ans = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            j2 = (j + rowShift[i]) % n
            i2 = (i + colShift[j2]) % n
            ans[i2][j2] = grid[i][j]
    return ans

assert shift_right_down(2, [[1, 2], [3, 4]], [1, 0], [0, 1]) == [[2, 4], [3, 1]]   # n = 2: trái = phải
assert shift_right_down(3, g2, [0, 0, 0], [1, 0, 0]) == [[7, 2, 3], [1, 5, 6], [4, 8, 9]]

# Bài 5(b)
def unshift(n, result, rowShift, colShift):
    cols = [list(c) for c in zip(*result)]
    cols = [c[-k:] + c[:-k] if k else c for c, k in zip(cols, colShift)]
    g = [list(r) for r in zip(*cols)]
    return [r[-k:] + r[:-k] if k else r for r, k in zip(g, rowShift)]

for n, grid, rs, cs, expected in examples:
    assert unshift(n, expected, rs, cs) == grid
random.seed(19)
for _ in range(100):
    n = random.randint(1, 8)
    grid, rs, cs = random_case(n)
    assert unshift(n, Solution().cyclicShift(n, grid, rs, cs), rs, cs) == grid
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (hai phép dịch vòng nối tiếp trên ma trận)

```python
class Solution:
    def cyclicShift(self, n: int, grid: List[List[int]], rowShift: List[int], colShift: List[int]) -> List[List[int]]:
        ans = [[0] * n for _ in range(n)]
        for i in range(n):
            for j in range(n):
                j2 = (j - rowShift[i]) % n          # dịch hàng i sang trái
                i2 = (i - colShift[j2]) % n         # dịch cột MỚI j2 lên trên
                ans[i2][j2] = grid[i][j]
        return ans
```

**Checklist**

- [ ] Làm **hàng trước, cột sau**; áp `colShift` cho **cột mới** `j2`, không phải cột cũ `j`?
- [ ] Dịch **trái** và **lên** dùng dấu `-`: `(j - k) % n`, `(i - k) % n`?
- [ ] Tạo ma trận bằng `[[0] * n for _ in range(n)]`, không dùng `[[0] * n] * n`?
- [ ] Không ghi đè dữ liệu chưa đọc (dùng ma trận mới)?
- [ ] Đã thử `n = 1`, mọi dịch bằng `0`, và ví dụ `n = 3`?
- [ ] Đã đối chiếu với mô phỏng từng bước (dịch 1 đơn vị lặp lại) trên input ngẫu nhiên?
- [ ] Skeleton đúng `class Solution` / `def cyclicShift(self, n: int, grid: List[List[int]], rowShift: List[int], colShift: List[int]) -> List[List[int]]`, có `from typing import List`?